# 06 — Blended LLM + ML

Combines XGBoost's calibrated probability scores with Llama-3.3-70b signal from Prompt Variance (05).  
**Goal:** beat both solo models on Charged Off F1 by exploiting their complementary strengths.

---

## Why blend?

| Model | Strength | Weakness |
|-------|----------|----------|
| XGBoost | Precise, stable, calibrated probabilities | Misses borderline defaults |
| LLM top_features_only | High Charged Off recall (87%) | Low precision (24%), 4x false alarms |

The blend wants XGBoost's precision filter **and** the LLM's broad default-detection net.

---

## Structure

| Part | What it does |
|------|--------------|
| 1 — Baselines | XGBoost solo vs all 5 LLM variants, side by side |
| 2 — Hard-label rules | Union / Intersection / Conditional overrides |
| 3 — Soft blend | alpha x threshold grid; heatmap of CO F1 across all combinations |
| 4 — Confidence gate | XGB decides alone when confident; LLM consulted only for borderlines |
| 5 — Description signal | Phase 5 desc-LLM as meta-feature (requires re-exporting 05 notebook) |
| 6 — Description risk scorer | **Strategy 5A**: LLM scores description text only → blend with XGBoost. **Strategy 5B**: LLM scores description + all structured features → blend with XGBoost. Tests whether borrower language adds signal beyond structured data. |

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

from llm_utils import (
    load_llm_sample,
    run_ml_on_sample,
    evaluate_predictions,
    RESULTS_DIR,
    DATA_DIR,
    MODEL_DIR,
)


def metrics_dict(name, y_true, y_pred):
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    return {
        'model':       name,
        'accuracy':    round(accuracy_score(yt, yp), 3),
        'precision_co': round(precision_score(yt, yp, pos_label=0, zero_division=0), 3),
        'recall_co':   round(recall_score(yt, yp, pos_label=0, zero_division=0), 3),
        'f1_co':       round(f1_score(yt, yp, pos_label=0, zero_division=0), 3),
    }


print('Imports OK.')
print(f'DATA_DIR:    {DATA_DIR}')
print(f'MODEL_DIR:   {MODEL_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')

### Train / reload XGBoost

If `xgb_model.joblib` and `thresholds.joblib` are missing, this cell trains XGBoost from `02_processed_data.npz` (~2-4 min) and saves the files.  
If the files already exist, it loads them and skips training.

In [ ]:
import xgboost as xgb

xgb_model_path  = os.path.join(MODEL_DIR, 'xgb_model.joblib')
thresh_path     = os.path.join(MODEL_DIR, 'thresholds.joblib')

if os.path.exists(xgb_model_path) and os.path.exists(thresh_path):
    print('Model files found — loading...')
    _model = joblib.load(xgb_model_path)
    xgb_threshold = joblib.load(thresh_path)['xgb']
    print(f'  XGBoost loaded  |  threshold = {xgb_threshold:.3f}')

else:
    print('Model files not found — training XGBoost from 02_processed_data.npz ...')
    print('(This takes ~2-4 minutes)')

    npz = np.load(os.path.join(DATA_DIR, '02_processed_data.npz'))
    X_train = npz['X_train']
    y_train = npz['y_train'].astype(int)
    X_val   = npz['X_val']
    y_val   = npz['y_val'].astype(int)

    print(f'  Train: {len(X_train):,} loans  |  {(y_train==1).sum():,} FP  {(y_train==0).sum():,} CO')
    print(f'  Val:   {len(X_val):,} loans')

    _model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='aucpr',
        early_stopping_rounds=20,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    _model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'  Training done.  Best iteration: {_model.best_iteration}')

    val_probs = _model.predict_proba(X_val)[:, 1]
    best_f1, xgb_threshold = 0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        f1 = f1_score(y_val, (val_probs >= t).astype(int), pos_label=0, zero_division=0)
        if f1 > best_f1:
            best_f1, xgb_threshold = f1, round(float(t), 2)
    print(f'  Optimal threshold: {xgb_threshold:.3f}  (val CO F1: {best_f1:.3f})')

    os.makedirs(MODEL_DIR, exist_ok=True)
    joblib.dump(_model,               xgb_model_path)
    joblib.dump({'xgb': xgb_threshold}, thresh_path)
    print(f'  Saved to {MODEL_DIR}')

In [ ]:
# Ground-truth sample
llm_sample = load_llm_sample()
y_true     = llm_sample['loan_status'].values
print(f'Sample: {len(llm_sample)} loans  |  FP: {(y_true==1).sum()}  |  CO: {(y_true==0).sum()}')

# XGBoost predictions on the same sample
xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)
print(f'XGBoost: threshold={xgb_threshold:.3f}  |  predicts {(xgb_preds==0).sum()} CO, {(xgb_preds==1).sum()} FP')

# LLM predictions from 05_Prompt_Variance
preds_df   = pd.read_csv(os.path.join(RESULTS_DIR, '05_predictions.csv'))
print(f'\nLLM predictions loaded: {len(preds_df)} rows | phases {sorted(preds_df["phase"].unique())}')

p1 = preds_df[preds_df['phase'] == 1]
LLM_VARIANTS = {}
for vname, grp in p1.groupby('variant'):
    LLM_VARIANTS[vname] = grp.sort_values('loan_index')['prediction'].astype(int).values

# Auto-detect the best variant from Phase 1 metrics saved by nb05.
# Avoids stale hardcoding when the winner changes after a re-run with new variants/sample.
_p1_metrics_path = os.path.join(RESULTS_DIR, '05_phase1_metrics.csv')
if os.path.exists(_p1_metrics_path):
    _p1_m = pd.read_csv(_p1_metrics_path, index_col=0)
    _best_candidate = _p1_m['f1_charged_off'].idxmax()
    if _best_candidate in LLM_VARIANTS:
        BEST_LLM = _best_candidate
        print(f'Best LLM variant (auto-detected from 05_phase1_metrics.csv): {BEST_LLM}')
    else:
        BEST_LLM = 'top_features_only'
        print(f'WARNING: auto-detected winner "{_best_candidate}" not in LLM_VARIANTS — falling back to {BEST_LLM}')
else:
    BEST_LLM = 'top_features_only'
    print(f'05_phase1_metrics.csv not found — using fallback: {BEST_LLM}')

llm_best = LLM_VARIANTS[BEST_LLM]

print(f'LLM variants loaded: {sorted(LLM_VARIANTS.keys())}')
print(f'Winner from Phase 1: {BEST_LLM}')

# Phase 5 (with desc) availability check
HAS_DESC = len(preds_df[preds_df['phase'] == 5]) > 0
print(f'Phase 5 desc data: {"available" if HAS_DESC else "NOT FOUND — re-run export cell in 05_Prompt_Variance.ipynb"}')

---
## Part 1 — Baselines
*How does XGBoost compare to each LLM variant on the same 100-loan sample?*

In [ ]:
print('XGBoost:')
evaluate_predictions(y_true, xgb_preds.tolist(), label='XGBoost')

print('\n' + '='*60)
print('LLM variants (Phase 1, no description):')
for vname in sorted(LLM_VARIANTS):
    evaluate_predictions(y_true, LLM_VARIANTS[vname].tolist(), label=f'LLM/{vname}')

# Summary table
baseline_rows = [metrics_dict('XGBoost', y_true, xgb_preds)]
for vname in sorted(LLM_VARIANTS):
    baseline_rows.append(metrics_dict(f'LLM/{vname}', y_true, LLM_VARIANTS[vname]))

baseline_df = pd.DataFrame(baseline_rows).set_index('model')
print('\n--- Summary ---')
print(baseline_df.to_string())

xgb_m = metrics_dict('XGBoost', y_true, xgb_preds)
llm_m = metrics_dict(BEST_LLM,  y_true, llm_best)
print(f'\nXGBoost CO F1: {xgb_m["f1_co"]:.3f}  |  precision: {xgb_m["precision_co"]:.3f}  |  recall: {xgb_m["recall_co"]:.3f}')
print(f'LLM best CO F1: {llm_m["f1_co"]:.3f}  |  precision: {llm_m["precision_co"]:.3f}  |  recall: {llm_m["recall_co"]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Part 1 — Solo Model Baselines (100-loan sample)', fontsize=12)
colors = plt.cm.tab10.colors

models = baseline_df.index.tolist()
x = np.arange(len(models))
cols_palette = ['#e74c3c'] + [colors[i % 9] for i in range(1, len(models) + 1)]

for ax, (col, title) in zip(axes, [('f1_co', 'Charged Off F1  (primary)'), ('accuracy', 'Overall Accuracy')]):
    vals = baseline_df[col].values
    bars = ax.bar(x, vals, color=cols_palette[:len(models)], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.008,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_title(title)
    ax.axhline(0.5, color='grey', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_baselines.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2 — Hard-Label Ensemble Rules
*Simple logical combinations of XGBoost and best-LLM predictions.*

| Rule | Logic | Expected effect |
|------|-------|-----------------|
| Union | CO if XGB **or** LLM says CO | Max recall, lower precision |
| Intersection | CO only if **both** say CO | Max precision, lower recall |
| XGB primary + LLM borderline boost | XGB base; flip borderline FP to CO when LLM says CO | Adds LLM's missed defaults at low XGB-confidence |
| LLM primary + XGB high-conf prune | LLM base; flip CO to FP when XGB is very confident FP | Prunes LLM false alarms |

In [ ]:
ens_rows = [
    metrics_dict('XGBoost (solo)',         y_true, xgb_preds),
    metrics_dict(f'LLM/{BEST_LLM} (solo)', y_true, llm_best),
]

# Union: CO if EITHER says CO
union_preds = np.where((xgb_preds == 0) | (llm_best == 0), 0, 1)
ens_rows.append(metrics_dict('Union  (XGB OR LLM)', y_true, union_preds))

# Intersection: CO only if BOTH say CO
inter_preds = np.where((xgb_preds == 0) & (llm_best == 0), 0, 1)
ens_rows.append(metrics_dict('Intersection  (XGB AND LLM)', y_true, inter_preds))

# XGB primary: flip XGB's borderline FP->CO when LLM says CO
band = 0.15
borderline = (xgb_probs > (xgb_threshold - band)) & (xgb_probs < (xgb_threshold + band))
xgb_boost = xgb_preds.copy()
xgb_boost[(xgb_preds == 1) & (llm_best == 0) & borderline] = 0
ens_rows.append(metrics_dict(f'XGB primary + LLM borderline boost (+/-{band})', y_true, xgb_boost))

# LLM primary: XGB very-confident FP overrides LLM's CO
llm_prune = llm_best.copy()
llm_prune[(llm_best == 0) & (xgb_probs >= (xgb_threshold + 0.25))] = 1
ens_rows.append(metrics_dict('LLM primary + XGB high-conf prune', y_true, llm_prune))

ens_df = pd.DataFrame(ens_rows).set_index('model')
print('Hard-label ensemble rules:')
print(ens_df.to_string())
print(f'\nBest CO F1 so far: {ens_df["f1_co"].max():.3f}  ({ens_df["f1_co"].idxmax()})')

---
## Part 3 — Soft Blend
*Sweep blend weight alpha and decision threshold together.*

```
score = alpha * P_xgb(FP) + (1 - alpha) * llm_pred
predict FP  if score >= threshold
predict CO  otherwise
```

- `alpha = 1` → XGBoost only (LLM ignored)
- `alpha = 0` → LLM only (XGBoost ignored)
- Optimal `alpha` shows how much XGBoost vs LLM is worth in the blend

In [ ]:
alphas     = np.round(np.arange(0.00, 1.01, 0.05), 2)
thresh_vals = np.round(
    np.unique(np.append(np.arange(0.05, 0.96, 0.05), xgb_threshold)), 3
)

llm_float = llm_best.astype(float)  # 0.0=CO, 1.0=FP

grid_rows = []
for a in alphas:
    for t in thresh_vals:
        score = a * xgb_probs + (1.0 - a) * llm_float
        preds = (score >= t).astype(int)
        grid_rows.append({
            'alpha':      a,
            'threshold':  t,
            'f1_co':      f1_score(y_true, preds, pos_label=0, zero_division=0),
            'recall_co':  recall_score(y_true, preds, pos_label=0, zero_division=0),
            'prec_co':    precision_score(y_true, preds, pos_label=0, zero_division=0),
            'accuracy':   accuracy_score(y_true, preds),
        })

grid_df  = pd.DataFrame(grid_rows)
best_row = grid_df.loc[grid_df['f1_co'].idxmax()]

best_score       = best_row['alpha'] * xgb_probs + (1.0 - best_row['alpha']) * llm_float
best_blend_preds = (best_score >= best_row['threshold']).astype(int)

print(f'Grid size: {len(grid_df)} combinations  ({len(alphas)} alphas x {len(thresh_vals)} thresholds)')
print(f'\nOptimal soft blend:')
print(f'  alpha={best_row["alpha"]:.2f}  (XGB weight)  |  threshold={best_row["threshold"]:.2f}')
print(f'  CO F1:      {best_row["f1_co"]:.3f}')
print(f'  Recall CO:  {best_row["recall_co"]:.3f}')
print(f'  Prec CO:    {best_row["prec_co"]:.3f}')
print(f'  Accuracy:   {best_row["accuracy"]:.3f}')
print(f'\n  XGBoost solo F1: {xgb_m["f1_co"]:.3f}')
print(f'  LLM solo F1:     {llm_m["f1_co"]:.3f}')
print(f'  Blend gain over XGB: {best_row["f1_co"] - xgb_m["f1_co"]:+.3f}')
print(f'  Blend gain over LLM: {best_row["f1_co"] - llm_m["f1_co"]:+.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Part 3 — Soft Blend  (LLM: {BEST_LLM})', fontsize=12)

# --- Heatmap: alpha x threshold -> CO F1 ---
pivot = grid_df.pivot_table(index='alpha', columns='threshold', values='f1_co')
ax = axes[0]
im = ax.imshow(pivot.values, aspect='auto', origin='lower',
               cmap='RdYlGn', vmin=0.1, vmax=0.75)
plt.colorbar(im, ax=ax, label='Charged Off F1', shrink=0.85)
cols_idx = list(pivot.columns)
rows_idx  = list(pivot.index)
step_c = max(1, len(cols_idx) // 10)
step_r = max(1, len(rows_idx) // 10)
ax.set_xticks(range(0, len(cols_idx), step_c))
ax.set_xticklabels([f'{c:.2f}' for c in cols_idx[::step_c]], rotation=45, fontsize=7)
ax.set_yticks(range(0, len(rows_idx), step_r))
ax.set_yticklabels([f'{r:.2f}' for r in rows_idx[::step_r]], fontsize=7)
ax.set_xlabel('Decision threshold  (lower = flag more CO)')
ax.set_ylabel('alpha  (0 = LLM only  |  1 = XGB only)')
ax.set_title('CO F1  (*= optimum)')
best_ai = rows_idx.index(best_row['alpha'])
best_ti = cols_idx.index(best_row['threshold'])
ax.scatter([best_ti], [best_ai], color='white', s=200, zorder=5, marker='*')

# --- Alpha curve: best F1 per alpha (max over threshold) ---
alpha_best = grid_df.groupby('alpha')['f1_co'].max().reset_index()
ax2 = axes[1]
ax2.plot(alpha_best['alpha'], alpha_best['f1_co'], 'b-o', markersize=5, label='Best blend F1')
ax2.axhline(xgb_m['f1_co'],  color='steelblue', linestyle='--', alpha=0.7,
            label=f'XGB solo ({xgb_m["f1_co"]:.3f})')
ax2.axhline(llm_m['f1_co'],  color='#f4a529',   linestyle='--', alpha=0.7,
            label=f'LLM solo ({llm_m["f1_co"]:.3f})')
ax2.axvline(best_row['alpha'], color='red', linestyle=':', alpha=0.7,
            label=f'Optimal alpha={best_row["alpha"]:.2f}')
ax2.set_xlabel('alpha  (XGBoost weight)')
ax2.set_ylabel('Best achievable CO F1  (max over threshold)')
ax2.set_title('Marginal value of LLM signal')
ax2.set_ylim(0, 0.9)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_soft_blend_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Which LLM variant blends best with XGBoost?
print('Best blend F1 per LLM variant (optimised over alpha and threshold):')
print(f'  XGBoost solo F1: {xgb_m["f1_co"]:.3f}\n')

blend_var_rows = []
for vname in sorted(LLM_VARIANTS):
    vfloat  = LLM_VARIANTS[vname].astype(float)
    solo_f1 = f1_score(y_true, LLM_VARIANTS[vname], pos_label=0, zero_division=0)
    best_f1_v, best_a_v, best_t_v = 0, 0, 0.5
    for a in alphas:
        for t in thresh_vals:
            sc = a * xgb_probs + (1.0 - a) * vfloat
            f1 = f1_score(y_true, (sc >= t).astype(int), pos_label=0, zero_division=0)
            if f1 > best_f1_v:
                best_f1_v, best_a_v, best_t_v = f1, a, t
    gain = best_f1_v - xgb_m['f1_co']
    flag = '  <<< best blend partner' if vname == BEST_LLM else ''
    print(f'  {vname:<22}  solo={solo_f1:.3f}  blend={best_f1_v:.3f}  gain over XGB={gain:+.3f}  (a={best_a_v:.2f}, t={best_t_v:.2f}){flag}')
    blend_var_rows.append({'variant': vname, 'solo_f1': solo_f1, 'blend_f1': best_f1_v,
                            'gain_over_xgb': gain, 'best_alpha': best_a_v, 'best_thresh': best_t_v})

blend_var_df = pd.DataFrame(blend_var_rows)
print(f'\nBest blending variant: {blend_var_df.loc[blend_var_df["blend_f1"].idxmax(), "variant"]}  (F1={blend_var_df["blend_f1"].max():.3f})')

---
## Part 4 — Confidence-Gated Consultation
*XGBoost decides alone when its score is far from the decision boundary. The LLM is only consulted for borderline loans.*

```
P_xgb < (threshold - band)  ->  CO  (XGB very confident)
P_xgb >= (threshold + band)  ->  FP  (XGB very confident)
otherwise                    ->  LLM prediction
```

**band = 0** means XGB decides everything (0% LLM calls).  
**band = threshold** means everything in [0, 2*threshold] is uncertain — maximum LLM involvement.  

We sweep band width and show the F1/consultation-rate tradeoff.

In [ ]:
band_widths = np.round(np.arange(0.00, 0.511, 0.01), 3)
gate_rows   = []

for band in band_widths:
    lo = max(0.0, xgb_threshold - band)
    hi = min(1.0, xgb_threshold + band)
    preds = np.empty(len(xgb_probs), dtype=int)
    n_llm = 0
    for i in range(len(xgb_probs)):
        p = xgb_probs[i]
        if p < lo:
            preds[i] = 0
        elif p >= hi:
            preds[i] = 1
        else:
            preds[i] = int(llm_best[i])
            n_llm += 1
    gate_rows.append({
        'band':        band,
        'pct_llm':     round(n_llm / len(xgb_probs) * 100, 1),
        'f1_co':       round(f1_score(y_true, preds, pos_label=0, zero_division=0), 4),
        'recall_co':   round(recall_score(y_true, preds, pos_label=0, zero_division=0), 4),
        'prec_co':     round(precision_score(y_true, preds, pos_label=0, zero_division=0), 4),
        'accuracy':    round(accuracy_score(y_true, preds), 4),
    })

gate_df   = pd.DataFrame(gate_rows)
best_gate = gate_df.loc[gate_df['f1_co'].idxmax()]

# Store best gate predictions
lo_opt = max(0.0, xgb_threshold - best_gate['band'])
hi_opt = min(1.0, xgb_threshold + best_gate['band'])
gate_preds_best = np.array([
    0 if xgb_probs[i] < lo_opt else (1 if xgb_probs[i] >= hi_opt else int(llm_best[i]))
    for i in range(len(xgb_probs))
])

print(f'Swept {len(band_widths)} band widths around threshold={xgb_threshold:.3f}')
print(f'\nOptimal confidence gate:')
print(f'  Band:          +/- {best_gate["band"]:.3f}')
print(f'  Uncertain zone: [{lo_opt:.3f}, {hi_opt:.3f})')
print(f'  LLM consulted:  {best_gate["pct_llm"]:.1f}% of loans')
print(f'  CO F1:          {best_gate["f1_co"]:.3f}')
print(f'  Recall CO:      {best_gate["recall_co"]:.3f}')
print(f'  Precision CO:   {best_gate["prec_co"]:.3f}')
print(f'  Accuracy:       {best_gate["accuracy"]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Part 4 — Confidence-Gated Consultation  (XGB threshold={xgb_threshold:.3f})', fontsize=12)

ax1, ax2 = axes

ax1.plot(gate_df['band'], gate_df['f1_co'],    'b-o', markersize=3, label='CO F1')
ax1.plot(gate_df['band'], gate_df['recall_co'], 'r-^', markersize=3, label='CO Recall')
ax1.plot(gate_df['band'], gate_df['prec_co'],   'g-s', markersize=3, label='CO Precision')
ax1.plot(gate_df['band'], gate_df['accuracy'],  'k-', markersize=2, alpha=0.5, label='Accuracy')
ax1.axhline(xgb_m['f1_co'],  color='steelblue', linestyle='--', alpha=0.6, label=f'XGB solo F1={xgb_m["f1_co"]:.3f}')
ax1.axhline(llm_m['f1_co'],  color='#f4a529',   linestyle='--', alpha=0.6, label=f'LLM solo F1={llm_m["f1_co"]:.3f}')
ax1.axvline(best_gate['band'], color='red', linestyle=':', alpha=0.7)
ax1.set_xlabel('Band width (+/-)')
ax1.set_ylabel('Score')
ax1.set_title('Metrics vs Uncertainty Band Width')
ax1.legend(fontsize=7)
ax1.set_ylim(0, 1.05)

ax2.plot(gate_df['band'], gate_df['pct_llm'], 'r-D', markersize=3)
ax2.axvline(best_gate['band'], color='red', linestyle=':', alpha=0.7,
            label=f'Optimal band={best_gate["band"]:.3f} -> {best_gate["pct_llm"]:.1f}% LLM calls')
ax2.set_xlabel('Band width (+/-)')
ax2.set_ylabel('% loans sent to LLM')
ax2.set_title('LLM Consultation Rate  (API cost proxy)')
ax2.set_ylim(0, 105)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_confidence_gate.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5 — Description Signal as Meta-Feature
*Does the LLM's prediction change when given the borrower description tell us something XGBoost can't see?*

When the LLM flips its prediction after reading the description AND that flip is toward the correct answer, the description carries genuine signal.  
We test whether blending XGBoost with the **description-aware LLM** (Phase 5) beats blending with Phase 1 (no-desc LLM).

> **Requires:** re-running the Export cell in `05_Prompt_Variance.ipynb` first — Phase 5 rows must be present in `05_predictions.csv`.

In [ ]:
desc_blend_preds = None
desc_blend_name  = None

if not HAS_DESC:
    print('[SKIP] Phase 5 data not found in 05_predictions.csv.')
    print('Steps to enable:')
    print('  1. Open 05_Prompt_Variance.ipynb')
    print('  2. Re-run the "Export Results" cell (cell-export)')
    print('  3. Re-run the load-data cell above (nb06-load-data)')
else:
    p5  = preds_df[preds_df['phase'] == 5].copy()
    p5['variant_base'] = p5['variant'].str.replace('_with_desc', '', regex=False)
    p1w = preds_df[(preds_df['phase'] == 1) & (preds_df['variant'] == BEST_LLM)].copy()

    desc_df = p1w[['loan_index', 'actual', 'prediction']].merge(
        p5[p5['variant_base'] == BEST_LLM][['loan_index', 'prediction']].rename(
            columns={'prediction': 'pred_desc'}
        ),
        on='loan_index', how='inner'
    ).sort_values('loan_index').reset_index(drop=True)

    desc_df['xgb_prob'] = xgb_probs
    desc_df['desc_changed'] = (desc_df['prediction'] != desc_df['pred_desc']).astype(int)
    desc_df['desc_to_co']   = ((desc_df['prediction'] == 1) & (desc_df['pred_desc'] == 0)).astype(int)
    desc_df['desc_to_fp']   = ((desc_df['prediction'] == 0) & (desc_df['pred_desc'] == 1)).astype(int)
    desc_df['desc_helped']  = (
        (desc_df['prediction'] != desc_df['actual']) & (desc_df['pred_desc'] == desc_df['actual'])
    ).astype(int)
    desc_df['desc_hurt'] = (
        (desc_df['prediction'] == desc_df['actual']) & (desc_df['pred_desc'] != desc_df['actual'])
    ).astype(int)

    changed = desc_df[desc_df['desc_changed'] == 1]
    print(f'Variant: {BEST_LLM}')
    print(f'  Loans where description changed prediction: {len(changed)}/100')
    print(f'    Changed to CO (FP->CO): {desc_df["desc_to_co"].sum()}')
    print(f'    Changed to FP (CO->FP): {desc_df["desc_to_fp"].sum()}')
    if len(changed):
        print(f'  Among changes:')
        print(f'    Helped (moved toward correct): {desc_df["desc_helped"].sum()}  ({desc_df["desc_helped"].sum()/len(changed)*100:.0f}%)')
        print(f'    Hurt   (moved away):           {desc_df["desc_hurt"].sum()}  ({desc_df["desc_hurt"].sum()/len(changed)*100:.0f}%)')

    # Soft blend using desc-LLM predictions
    llm_desc_float = desc_df['pred_desc'].values.astype(float)
    grid_desc = []
    for a in alphas:
        for t in thresh_vals:
            sc = a * xgb_probs + (1.0 - a) * llm_desc_float
            grid_desc.append({
                'alpha': a, 'threshold': t,
                'f1_co': f1_score(y_true, (sc >= t).astype(int), pos_label=0, zero_division=0),
            })
    grid_desc_df = pd.DataFrame(grid_desc)
    best_desc_row = grid_desc_df.loc[grid_desc_df['f1_co'].idxmax()]
    best_desc_sc  = best_desc_row['alpha'] * xgb_probs + (1.0 - best_desc_row['alpha']) * llm_desc_float
    desc_blend_preds = (best_desc_sc >= best_desc_row['threshold']).astype(int)
    desc_blend_name  = f'Soft blend + desc  (a={best_desc_row["alpha"]:.2f}, t={best_desc_row["threshold"]:.2f})'

    print(f'\nSoft blend with description-LLM:')
    print(f'  alpha={best_desc_row["alpha"]:.2f}  threshold={best_desc_row["threshold"]:.2f}')
    print(f'  CO F1: {best_desc_row["f1_co"]:.3f}  (no-desc blend: {best_row["f1_co"]:.3f}  delta={best_desc_row["f1_co"]-best_row["f1_co"]:+.3f})')

---
## Part 6 — Description Risk Scorer (Strategy 5)
*LLM reads text and outputs a numeric risk score — not a binary prediction. Score is then blended with XGBoost.*

| Version | LLM input | Hypothesis |
|---------|-----------|------------|
| **5A — Desc only** | Borrower description text only | Does borrower language carry signal XGBoost never saw? |
| **5B — Desc + Context** | Description + all 21 structured features | Does description add marginal value when the model already knows the numbers? |

**Blend formula** (same alpha × threshold grid as Part 3):
```
p_fp_desc  =  1 - (risk_score / 10)          # convert 0-10 risk to P(Fully Paid) analog
blend_score = alpha * P_xgb(FP) + (1-alpha) * p_fp_desc
predict FP  if blend_score >= threshold
```

Loans with no description → **5A** gets neutral score 5.0; **5B** rates from structured features alone.

> Requires ~200 NVIDIA NIM API calls (~2-5 min). Scored at `temperature=0`.

In [ ]:
import re as _re
import time as _time_setup
from openai import OpenAI as _OAI

from llm_utils import load_api_key, format_loan_features

_NIM_KEY   = load_api_key('nvidia')
_NIM_URL   = 'https://integrate.api.nvidia.com/v1'
_NIM_MODEL = 'meta/llama-3.3-70b-instruct'
_nim       = _OAI(api_key=_NIM_KEY, base_url=_NIM_URL)

_SCORE_GUIDE = (
    '0-3 = low risk  (specific purpose, professional tone, clear repayment plan)\n'
    '4-6 = moderate risk  (vague, generic, or neutral)\n'
    '7-10 = high risk  (desperate tone, inconsistent story, alarming language)'
)
_JSON_INSTR = 'Reply with ONLY valid JSON, no other text:\n{"risk_score": <number 0-10>, "reasoning": "<one sentence>"}'

# ── Best-of-N configuration ───────────────────────────────────────────────────
# CME295 Lec 6 (test-time compute / self-consistency, Wang et al. 2022):
# sample N responses at low temperature, average the numeric scores.
# This reduces single-call variance without any training signal — pure inference-time scaling.
# N=3 at temp=0.3: small ensemble, moderate diversity, ~3x API cost vs N=1.
BON_N    = 3
BON_TEMP = 0.3


def _nim_call(prompt: str, max_retries: int = 6, temperature: float = 0.0) -> str:
    for attempt in range(max_retries):
        try:
            resp = _nim.chat.completions.create(
                model=_NIM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=80,
                temperature=temperature,
            )
            return resp.choices[0].message.content or ''
        except Exception as e:
            err = str(e).lower()
            if '429' in str(e) or 'rate' in err or '503' in str(e) or 'connection' in err:
                wait = 2 ** attempt * 5
                print(f'  NIM retry {attempt+1}/{max_retries} in {wait}s...')
                _time_setup.sleep(wait)
            else:
                raise
    raise RuntimeError('NIM API failed after retries')


def _parse_risk_score(text: str) -> tuple:
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
        text = text.strip()
    try:
        obj = json.loads(text)
        return min(10.0, max(0.0, float(obj.get('risk_score', 5.0)))), str(obj.get('reasoning', ''))
    except Exception:
        pass
    m = _re.search(r'"risk_score"\s*:\s*([0-9]+(?:\.[0-9]+)?)', text)
    if m:
        return min(10.0, max(0.0, float(m.group(1)))), ''
    return 5.0, 'PARSE_ERROR'


def _nim_call_bon(prompt: str) -> tuple:
    """Best-of-N: call BON_N times at BON_TEMP, return (avg_score, reasoning from median-score call).

    Averaging scores across N stochastic samples reduces single-call variance —
    the same principle as majority-vote self-consistency (Wang et al., 2022).
    """
    scores, reasonings = [], []
    for _ in range(BON_N):
        raw = _nim_call(prompt, temperature=BON_TEMP)
        s, r = _parse_risk_score(raw)
        scores.append(s)
        reasonings.append(r)
    avg   = float(np.mean(scores))
    # Return the reasoning from the call whose score was closest to the average
    m_idx = int(np.argmin(np.abs(np.array(scores) - avg)))
    return avg, reasonings[m_idx]


def score_5a_desc_only(desc) -> tuple:
    """Version 5A: LLM sees ONLY the description. Returns (score 0-10, reasoning, has_desc).
    Uses Best-of-N (N=BON_N) averaging at temperature=BON_TEMP for stability."""
    if not desc or (isinstance(desc, float) and np.isnan(desc)) or str(desc).strip() == '':
        return 5.0, 'no description', False
    prompt = (
        'You are a credit risk analyst. Your ONLY input is the borrower\'s loan description.\n\n'
        f'Description: "{str(desc).strip()}"\n\n'
        f'Rate the default risk 0-10.\n{_SCORE_GUIDE}\n\n{_JSON_INSTR}'
    )
    score, reasoning = _nim_call_bon(prompt)
    return score, reasoning, True


def score_5b_desc_with_context(row: pd.Series) -> tuple:
    """Version 5B: LLM sees description + all structured features. Returns (score, reasoning, has_desc).
    Uses Best-of-N (N=BON_N) averaging at temperature=BON_TEMP for stability."""
    desc     = row.get('desc', '')
    has_desc = bool(desc and not (isinstance(desc, float) and np.isnan(desc)) and str(desc).strip())
    features = format_loan_features(row, include_desc=False)
    desc_line = f'\nBorrower description:\n  "{str(desc).strip()}"' if has_desc else '\n(No description provided.)'
    prompt = (
        'You are a credit risk analyst reviewing a full loan application.\n\n'
        f'Loan features:\n{features}{desc_line}\n\n'
        'Rate the overall default risk 0-10. Focus on how the description aligns with or contradicts the structured data.\n'
        f'{_SCORE_GUIDE}\n\n{_JSON_INSTR}'
    )
    score, reasoning = _nim_call_bon(prompt)
    return score, reasoning, has_desc


print(f'Scorer functions ready (Best-of-N: N={BON_N}, temp={BON_TEMP}).  NIM model: {_NIM_MODEL}')
print(f'API key loaded: {"yes" if _NIM_KEY else "NO KEY FOUND — check .env"}')
print(f'Note: each loan now requires {BON_N} × 2 = {BON_N * 2} NIM calls ({BON_N}x cost vs single-call).')

In [ ]:
import time as _time

print(f'Scoring {len(llm_sample)} loans — ~200 NIM API calls total (~2-5 min).\n')

score_rows = []
start_t    = _time.time()

for i, (_, row) in enumerate(llm_sample.iterrows()):
    entry = {'loan_index': i, 'actual': int(row['loan_status'])}

    # Version 5A: description only
    try:
        s5a, r5a, h5a = score_5a_desc_only(row.get('desc', ''))
    except Exception as exc:
        s5a, r5a, h5a = 5.0, f'ERROR: {exc}', False
    entry.update({'score_5a': s5a, 'reasoning_5a': r5a, 'has_desc': h5a})

    # Version 5B: description + all structured features
    try:
        s5b, r5b, _ = score_5b_desc_with_context(row)
    except Exception as exc:
        s5b, r5b = 5.0, f'ERROR: {exc}'
    entry.update({'score_5b': s5b, 'reasoning_5b': r5b})

    score_rows.append(entry)

    if (i + 1) % 10 == 0:
        elapsed = _time.time() - start_t
        eta     = elapsed / (i + 1) * (len(llm_sample) - i - 1)
        print(f'  {i+1:3d}/100  5A={s5a:.1f}  5B={s5b:.1f}  desc={"yes" if h5a else "no "}  |  {elapsed:.0f}s elapsed ~{eta:.0f}s remaining')

scores_df      = pd.DataFrame(score_rows)
scores_5a_vals = scores_df['score_5a'].values.astype(float)
scores_5b_vals = scores_df['score_5b'].values.astype(float)
has_desc_mask  = scores_df['has_desc'].values.astype(bool)

n_with_desc    = has_desc_mask.sum()
elapsed_total  = _time.time() - start_t
print(f'\nDone in {elapsed_total:.0f}s.')
print(f'  Loans with description: {n_with_desc}/100')
print(f'  5A — mean={scores_5a_vals.mean():.2f}  std={scores_5a_vals.std():.2f}  range=[{scores_5a_vals.min():.1f}, {scores_5a_vals.max():.1f}]')
print(f'  5B — mean={scores_5b_vals.mean():.2f}  std={scores_5b_vals.std():.2f}  range=[{scores_5b_vals.min():.1f}, {scores_5b_vals.max():.1f}]')

# Correlation: how much novel signal does each version carry vs XGBoost?
corr_ab    = np.corrcoef(scores_5a_vals, scores_5b_vals)[0, 1]
corr_a_xgb = np.corrcoef(scores_5a_vals, 1 - xgb_probs)[0, 1]  # 1-P(FP) = P(CO)
corr_b_xgb = np.corrcoef(scores_5b_vals, 1 - xgb_probs)[0, 1]
print(f'\nCorrelation with XGBoost P(CO):')
print(f'  5A (desc only)      r={corr_a_xgb:.3f}  — low = novel text signal beyond structured features')
print(f'  5B (desc+context)   r={corr_b_xgb:.3f}  — high = LLM replicates XGBoost when given same inputs')
print(f'  5A vs 5B            r={corr_ab:.3f}')

In [ ]:
# P(FP) analog from each scorer: high risk score -> low probability of Fully Paid
p_fp_5a = 1.0 - (scores_5a_vals / 10.0)
p_fp_5b = 1.0 - (scores_5b_vals / 10.0)

def _run_blend_grid(p_desc, label):
    rows = []
    for a in alphas:
        for t in thresh_vals:
            sc    = a * xgb_probs + (1.0 - a) * p_desc
            preds = (sc >= t).astype(int)
            rows.append({
                'alpha':     a,
                'threshold': t,
                'f1_co':     f1_score(y_true, preds, pos_label=0, zero_division=0),
                'recall_co': recall_score(y_true, preds, pos_label=0, zero_division=0),
                'prec_co':   precision_score(y_true, preds, pos_label=0, zero_division=0),
                'accuracy':  accuracy_score(y_true, preds),
            })
    df   = pd.DataFrame(rows)
    best = df.loc[df['f1_co'].idxmax()]
    best_sc    = best['alpha'] * xgb_probs + (1.0 - best['alpha']) * p_desc
    best_preds = (best_sc >= best['threshold']).astype(int)
    print(f'\n{label}:')
    print(f'  Optimal: alpha={best["alpha"]:.2f}  threshold={best["threshold"]:.2f}')
    print(f'  CO F1:      {best["f1_co"]:.3f}  (XGBoost solo: {xgb_m["f1_co"]:.3f}  gain: {best["f1_co"]-xgb_m["f1_co"]:+.3f})')
    print(f'  CO Recall:  {best["recall_co"]:.3f}')
    print(f'  CO Prec:    {best["prec_co"]:.3f}')
    print(f'  Accuracy:   {best["accuracy"]:.3f}')
    return df, best, best_preds

print('Running alpha x threshold blend grids for both versions...')
grid_5a, best_5a, preds_5a = _run_blend_grid(p_fp_5a, '5A — Description Only + XGBoost')
grid_5b, best_5b, preds_5b = _run_blend_grid(p_fp_5b, '5B — Description + Context + XGBoost')

# Desc-subset analysis: only loans that actually had a description
if has_desc_mask.sum() > 5:
    print(f'\n--- Desc-subset analysis ({has_desc_mask.sum()} loans with descriptions) ---')
    yt_sub = y_true[has_desc_mask]
    for lbl_s, preds_s in [
        ('5A blend', preds_5a[has_desc_mask]),
        ('5B blend', preds_5b[has_desc_mask]),
        ('XGB solo', xgb_preds[has_desc_mask]),
    ]:
        f1_s = f1_score(yt_sub, preds_s, pos_label=0, zero_division=0)
        print(f'  {lbl_s}: CO F1={f1_s:.3f}  (on desc-having loans only)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Part 6 — Description Risk Scorer: Strategy Comparison', fontsize=12)

# Panel 1: Alpha curves — 5A, 5B, and Part 3 soft blend
ax1 = axes[0]
for gdf, lbl, color in [
    (grid_5a, '5A: Desc only',              '#e91e63'),
    (grid_5b, '5B: Desc + context',         '#9c27b0'),
    (grid_df, 'Part 3 soft blend (full LLM)', '#4caf50'),
]:
    curve = gdf.groupby('alpha')['f1_co'].max()
    ax1.plot(curve.index, curve.values, '-o', markersize=4, label=lbl, color=color)
ax1.axhline(xgb_m['f1_co'], color='steelblue', linestyle='--', alpha=0.7,
            label=f'XGB solo ({xgb_m["f1_co"]:.3f})')
ax1.axhline(llm_m['f1_co'], color='#f4a529',   linestyle='--', alpha=0.7,
            label=f'LLM solo ({llm_m["f1_co"]:.3f})')
ax1.set_xlabel('alpha  (XGBoost weight)')
ax1.set_ylabel('Best CO F1  (max over threshold)')
ax1.set_title('Alpha Curve: All Score-Blend Variants')
ax1.legend(fontsize=7)
ax1.set_ylim(0, 0.9)

# Panel 2: Risk score distributions by actual label
ax2 = axes[1]
co_mask = (y_true == 0)
fp_mask = (y_true == 1)
ax2.hist(scores_5a_vals[co_mask], bins=10, alpha=0.6, color='red',      label='5A: Actual CO')
ax2.hist(scores_5a_vals[fp_mask], bins=10, alpha=0.6, color='steelblue', label='5A: Actual FP')
ax2.hist(scores_5b_vals[co_mask], bins=10, alpha=0.5, color='darkred',
         histtype='step', linewidth=2, label='5B: Actual CO')
ax2.hist(scores_5b_vals[fp_mask], bins=10, alpha=0.5, color='navy',
         histtype='step', linewidth=2, label='5B: Actual FP')
ax2.set_xlabel('LLM Risk Score  (0=safe, 10=risky)')
ax2.set_ylabel('Count')
ax2.set_title('Score Distribution by True Label')
ax2.legend(fontsize=7)

# Panel 3: 5A vs 5B scatter, coloured by actual label
ax3 = axes[2]
ax3.scatter(scores_5a_vals[fp_mask], scores_5b_vals[fp_mask],
            alpha=0.6, color='steelblue', label='Actual FP', s=30)
ax3.scatter(scores_5a_vals[co_mask], scores_5b_vals[co_mask],
            alpha=0.8, color='red', marker='^', label='Actual CO', s=40)
ax3.plot([0, 10], [0, 10], 'k--', alpha=0.3, label='y=x')
ax3.set_xlabel('5A score  (desc only)')
ax3.set_ylabel('5B score  (desc + context)')
ax3.set_title(f'5A vs 5B  (corr={corr_ab:.2f})')
ax3.legend(fontsize=7)
ax3.set_xlim(-0.5, 10.5)
ax3.set_ylim(-0.5, 10.5)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_desc_scorer.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Final Leaderboard
*All strategies ranked by Charged Off F1.*

In [ ]:
from IPython.display import display

blend_label = f'Soft blend  a={best_row["alpha"]:.2f}, t={best_row["threshold"]:.2f}'
gate_label  = f'Confidence gate  +/-{best_gate["band"]:.3f}  ({best_gate["pct_llm"]:.0f}% to LLM)'

final_rows = [
    metrics_dict('XGBoost (solo)',                      y_true, xgb_preds),
    metrics_dict(f'LLM/{BEST_LLM} (solo)',              y_true, llm_best),
    metrics_dict('Union  (XGB OR LLM)',                 y_true, union_preds),
    metrics_dict('Intersection  (XGB AND LLM)',         y_true, inter_preds),
    metrics_dict('XGB primary + LLM borderline boost',  y_true, xgb_boost),
    metrics_dict('LLM primary + XGB high-conf prune',   y_true, llm_prune),
    metrics_dict(blend_label,                           y_true, best_blend_preds),
    metrics_dict(gate_label,                            y_true, gate_preds_best),
]

if desc_blend_preds is not None:
    final_rows.append(metrics_dict(desc_blend_name, y_true, desc_blend_preds))

# Strategy 5A and 5B: description risk scorer
try:
    label_5a = f'5A: Desc-only scorer  (a={best_5a["alpha"]:.2f}, t={best_5a["threshold"]:.2f})'
    label_5b = f'5B: Desc+context scorer  (a={best_5b["alpha"]:.2f}, t={best_5b["threshold"]:.2f})'
    final_rows.append(metrics_dict(label_5a, y_true, preds_5a))
    final_rows.append(metrics_dict(label_5b, y_true, preds_5b))
except NameError:
    pass  # Part 6 cells not yet run

final_df = pd.DataFrame(final_rows).set_index('model').sort_values('f1_co', ascending=False)

display(
    final_df.style
    .format('{:.3f}')
    .highlight_max(color='lightgreen', axis=0)
    .highlight_min(color='#ffcccc',    axis=0)
    .set_caption('Sorted by Charged Off F1 (primary metric).  Green = best per column.')
)

winner = final_df.index[0]
print(f'\n=== WINNER: {winner} ===')
print(f'  CO F1:     {final_df.loc[winner, "f1_co"]:.3f}')
print(f'  Recall CO: {final_df.loc[winner, "recall_co"]:.3f}')
print(f'  Prec CO:   {final_df.loc[winner, "precision_co"]:.3f}')
print(f'  Accuracy:  {final_df.loc[winner, "accuracy"]:.3f}')
print(f'\n  vs XGBoost solo: {xgb_m["f1_co"]:.3f}  delta={final_df.loc[winner,"f1_co"]-xgb_m["f1_co"]:+.3f}')
print(f'  vs LLM solo:     {llm_m["f1_co"]:.3f}  delta={final_df.loc[winner,"f1_co"]-llm_m["f1_co"]:+.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
final_sorted = final_df.sort_values('f1_co')

def bar_color(name):
    if 'XGBoost (solo)' in name:  return '#2196f3'
    if 'LLM/' in name and '(' in name and 'blend' not in name.lower() and 'gate' not in name.lower(): return '#f4a529'
    return '#4caf50'

col_list = [bar_color(n) for n in final_sorted.index]
bars = ax.barh(final_sorted.index, final_sorted['f1_co'], color=col_list, alpha=0.85)
for bar, val in zip(bars, final_sorted['f1_co']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=8)
ax.set_xlabel('Charged Off F1  (primary metric)')
ax.set_title('Final Leaderboard\n(blue=XGB solo, orange=LLM solo, green=blend)', fontsize=11)
ax.set_xlim(0, final_sorted['f1_co'].max() + 0.08)
ax.axvline(xgb_m['f1_co'], color='steelblue', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_leaderboard.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

final_df.to_csv(os.path.join(RESULTS_DIR, '06_blend_results.csv'))
grid_df.to_csv(os.path.join(RESULTS_DIR, '06_soft_blend_grid.csv'),   index=False)
gate_df.to_csv(os.path.join(RESULTS_DIR, '06_gate_sweep.csv'),        index=False)
blend_var_df.to_csv(os.path.join(RESULTS_DIR, '06_blend_by_variant.csv'), index=False)

print('Saved:')
print(f'  06_blend_results.csv       -- final leaderboard ({len(final_df)} strategies)')
print(f'  06_soft_blend_grid.csv     -- alpha x threshold grid ({len(grid_df)} rows)')
print(f'  06_gate_sweep.csv          -- confidence gate sweep ({len(gate_df)} rows)')
print(f'  06_blend_by_variant.csv    -- best blend per LLM variant')

# Part 6 outputs (only if those cells have been run)
try:
    scores_df.to_csv(os.path.join(RESULTS_DIR, '06_desc_scores.csv'), index=False)
    grid_5a.to_csv(os.path.join(RESULTS_DIR, '06_desc_blend_grid_5a.csv'), index=False)
    grid_5b.to_csv(os.path.join(RESULTS_DIR, '06_desc_blend_grid_5b.csv'), index=False)
    print(f'  06_desc_scores.csv         -- per-loan 5A/5B scores + reasoning ({len(scores_df)} rows)')
    print(f'  06_desc_blend_grid_5a.csv  -- 5A blend grid ({len(grid_5a)} rows)')
    print(f'  06_desc_blend_grid_5b.csv  -- 5B blend grid ({len(grid_5b)} rows)')
    print(f'  06_desc_scorer.png')
except NameError:
    print('  (Part 6 outputs skipped — run Part 6 cells first)')

print(f'\nImages saved:')
print(f'  06_baselines.png')
print(f'  06_soft_blend_heatmap.png')
print(f'  06_confidence_gate.png')
print(f'  06_leaderboard.png')
print(f'\nAll in: {RESULTS_DIR}')

---
## Part 7 — Sentence Embeddings as XGBoost Features
*Can borrower description language improve credit risk detection beyond structured data?*

**CME295 inspiration:** Lectures 1–3 showed that pre-trained Transformer representations capture
semantic nuance invisible to hand-crafted features. The key insight: a model pre-trained on
billions of tokens has already learned associations between language patterns and outcomes that
no tabular feature can encode (e.g. "I need this to consolidate debt before losing my job" vs
"consolidating high-interest cards to free up cash").

**Experiment design:**
1. Encode each borrower description with `all-MiniLM-L6-v2` (sentence-transformers, 384-dim)
2. Reduce to 10 principal components via PCA — *unsupervised*, no labels used
3. Find which PCs correlate with default (r ≠ 0 → language signal exists)
4. Soft-blend the most-correlated PC with XGBoost's P(FP) and sweep the blend weight

| | XGBoost features | Description signal |
|--|--|--|
| Baseline | 74 structured features | None |
| + Embeddings | 74 structured features | PC1 of 384-dim description embedding |

**Honest caveat:** the blend grid is optimised on the same 76-loan eval sample — treat any gain
as an upper bound on true held-out performance, not a definitive improvement claim.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer as _ST
    _st_ok = True
except ImportError:
    print('[SKIP] sentence-transformers not installed.')
    print('       Install with:  pip install sentence-transformers')
    _st_ok = False

if _st_ok:
    from sklearn.decomposition import PCA as _PCA

    # 1. Encode borrower descriptions
    print('Loading all-MiniLM-L6-v2  (22M params, 384-dim output)...')
    _enc = _ST('all-MiniLM-L6-v2')

    _desc_col  = llm_sample['desc'] if 'desc' in llm_sample.columns else pd.Series([''] * len(llm_sample))
    _descs_raw = [str(d) if (pd.notna(d) and str(d).strip()) else '' for d in _desc_col]
    _has_text  = [bool(d.strip()) for d in _descs_raw]
    print(f'  Descriptions available: {sum(_has_text)}/{len(llm_sample)}')

    _emb384 = _enc.encode(_descs_raw, batch_size=32, show_progress_bar=False)   # (N, 384)

    # 2. PCA → 10 principal components  (unsupervised — no label signal used)
    _pca10   = _PCA(n_components=10, random_state=42)
    _pcs     = _pca10.fit_transform(_emb384)                                     # (N, 10)
    _var_exp = _pca10.explained_variance_ratio_.sum()
    print(f'  PCA 10-dim: {_var_exp:.1%} of embedding variance explained')

    # 3. Correlation analysis — any linguistic signal for default?
    print('\nCorrelation of description PCs with true label (negative r → PC predicts CO):')
    _r_vals = []
    for k in range(10):
        r = np.corrcoef(_pcs[:, k], y_true)[0, 1]
        _r_vals.append(r)
        flag = '  ← notable' if abs(r) > 0.10 else ''
        print(f'  PC{k+1:2d}: r={r:+.3f}{flag}')

    # 4. Soft blend: XGB P(FP) + most-correlated PC
    _best_k   = int(np.argmax(np.abs(_r_vals)))
    _best_pc  = _pcs[:, _best_k].copy()
    if _r_vals[_best_k] < 0:          # flip so high value = more likely FP
        _best_pc = -_best_pc
    _pc_min, _pc_max = _best_pc.min(), _best_pc.max()
    _pc_norm = (_best_pc - _pc_min) / (_pc_max - _pc_min + 1e-9)  # scale to [0, 1]

    _emb_rows = []
    for _w in np.round(np.arange(0.0, 0.31, 0.02), 2):
        for _t in thresh_vals:
            _sc = (1.0 - _w) * xgb_probs + _w * _pc_norm
            _pr = (_sc >= _t).astype(int)
            _emb_rows.append({
                'pc_weight': _w, 'threshold': _t,
                'f1_co':    f1_score(y_true, _pr, pos_label=0, zero_division=0),
                'accuracy': accuracy_score(y_true, _pr),
            })
    _emb_df   = pd.DataFrame(_emb_rows)
    _best_emb = _emb_df.loc[_emb_df['f1_co'].idxmax()]
    _best_sc  = (1.0 - _best_emb['pc_weight']) * xgb_probs + _best_emb['pc_weight'] * _pc_norm
    _emb_preds = (_best_sc >= _best_emb['threshold']).astype(int)

    print(f'\nBest XGB + PC{_best_k+1} blend (grid search in-sample on {len(llm_sample)} loans):')
    print(f'  PC weight={_best_emb["pc_weight"]:.2f}  threshold={_best_emb["threshold"]:.2f}')
    print(f'  CO F1:    {_best_emb["f1_co"]:.3f}  (XGB solo: {xgb_m["f1_co"]:.3f}'
          f'  delta={_best_emb["f1_co"]-xgb_m["f1_co"]:+.3f})')
    print(f'  Accuracy: {_best_emb["accuracy"]:.3f}')
    print(f'\n  Interpretation: delta={_best_emb["f1_co"]-xgb_m["f1_co"]:+.3f}.'
          f'  This is in-sample on {len(llm_sample)} loans — treat as an upper bound,')
    print(f'  not a held-out test result.  A positive delta means description language')
    print(f'  carries some rank-ordering signal complementary to structured features.')

    # 5. Visualise
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Part 7 — Sentence Embeddings (all-MiniLM-L6-v2 → PCA 10-dim)\n'
                 f'{_var_exp:.1%} embedding variance in 10 PCs', fontsize=11)

    ax1 = axes[0]
    _pc_raw = _pcs[:, _best_k]
    ax1.hist(_pc_raw[y_true == 1], bins=12, alpha=0.6, color='steelblue', label='Actual FP')
    ax1.hist(_pc_raw[y_true == 0], bins=12, alpha=0.6, color='red',       label='Actual CO')
    ax1.set_xlabel(f'PC{_best_k+1}  (most correlated with label, r={_r_vals[_best_k]:+.3f})')
    ax1.set_ylabel('Count')
    ax1.set_title(f'Description PC{_best_k+1} Distribution by True Label')
    ax1.legend()

    ax2 = axes[1]
    _curve = _emb_df.groupby('pc_weight')['f1_co'].max()
    ax2.plot(_curve.index, _curve.values, 'b-o', markersize=5)
    ax2.axhline(xgb_m['f1_co'], color='steelblue', linestyle='--', alpha=0.7,
                label=f'XGB solo ({xgb_m["f1_co"]:.3f})')
    ax2.set_xlabel(f'PC{_best_k+1} blend weight  (0 = XGB only)')
    ax2.set_ylabel('Best CO F1  (max over threshold)')
    ax2.set_title('Marginal Value of Description Embedding')
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '07_embeddings.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {RESULTS_DIR}/07_embeddings.png')